Read in downloaded medical abstracts

In [ ]:
from pandas import read_csv
abstracts = read_csv('../../data/abstracts.csv')

create hierarchy

In [2]:
brain_regions = [
    "Hippocampus", "Amygdala", "Thalamus", "Hypothalamus",
    "Caudate", "Putamen", "Globus Pallidus", "Substantia Nigra",
    "Cerebellum", "Prefrontal Cortex", "Motor Cortex",
    "Parietal Cortex", "Temporal Cortex", "Occipital Cortex",
    "Anterior Cingulate", "Posterior Cingulate", "Insula"
]

In [3]:
from nltk import *
ls = LancasterStemmer()

strip_regions = [ls.stem(w) for w in brain_regions]

In [4]:
from nltk.stem import WordNetLemmatizer
wnl = WordNetLemmatizer()

In [5]:
br_dict = {}
for b in brain_regions:
    br_dict[b] = [b, b.lower(),ls.stem(b)]

In [6]:
br_dict

{'Hippocampus': ['Hippocampus', 'hippocampus', 'hippocamp'],
 'Amygdala': ['Amygdala', 'amygdala', 'amygdal'],
 'Thalamus': ['Thalamus', 'thalamus', 'thalam'],
 'Hypothalamus': ['Hypothalamus', 'hypothalamus', 'hypothalam'],
 'Caudate': ['Caudate', 'caudate', 'caud'],
 'Putamen': ['Putamen', 'putamen', 'putam'],
 'Globus Pallidus': ['Globus Pallidus', 'globus pallidus', 'globus pallid'],
 'Substantia Nigra': ['Substantia Nigra',
  'substantia nigra',
  'substantia nigr'],
 'Cerebellum': ['Cerebellum', 'cerebellum', 'cerebell'],
 'Prefrontal Cortex': ['Prefrontal Cortex',
  'prefrontal cortex',
  'prefrontal cortex'],
 'Motor Cortex': ['Motor Cortex', 'motor cortex', 'motor cortex'],
 'Parietal Cortex': ['Parietal Cortex', 'parietal cortex', 'parietal cortex'],
 'Temporal Cortex': ['Temporal Cortex', 'temporal cortex', 'temporal cortex'],
 'Occipital Cortex': ['Occipital Cortex',
  'occipital cortex',
  'occipital cortex'],
 'Anterior Cingulate': ['Anterior Cingulate',
  'anterior cingu

for now, annotate whether it is brain related

In [ ]:
from nltk import *
strip_regions = ['hippocamp',
 'amygdal',
 'thalam',
 'hypothalam',
 'caud',
 'putam',
 'globus pallid',
 'substantia nigr',
 'cerebell',
 'prefrontal cortex',
 'motor cortex',
 'parietal cortex',
 'temporal cortex',
 'occipital cortex',
 'anterior cingulate',
 'posterior cingulate',
 'insul']
strip_regions.extend(brain_regions)

def related(token, regions):
    for r in regions.keys():
        if token in regions[r]:
            return("B-Brain")
    return("O")

def annotate_abstract(abstract):
    tokens = abstract.split()
    labels = [""] * len(tokens)
    for i, token in enumerate(tokens):
        labels[i] = related(ls.stem(token.lower()), br_dict)
        if ls.stem(token.lower()) in [br_dict[key] for key in br_dict.keys()]:
            print(token)
            labels[i] = "B-BRAIN" #brain region related
    return list(zip(tokens, labels))

#annotated_abstracts = [annotate_abstract(abstract) for abstract in list(abstracts['0'].dropna())]

In [ ]:
with open("annotated_brain_regions.conll", "w", encoding="utf-8") as f:
    for abstract in annotated_abstracts:
        for token, label in abstract:
            f.write(f"{token} {label}\n")
        f.write("\n")  # Separate sentences with a newline


brain region names

In [ ]:
# pip install biopython datasets

In [24]:
from Bio import Entrez
import requests
import xml.etree.ElementTree as ET

def get_brain_regions():
    url = "http://api.brain-map.org/api/v2/structure_graph_download/1.json"
    response = requests.get(url)
    data = response.json()
    
    brain_regions = []
    for structure in data["msg"]:
        print(name.lower())
        name = structure.get("name")
        if name:
            brain_regions.append(name.lower())
    return brain_regions


In [ ]:
Entrez.email = #the email address has to be set here

store pubmed

In [ ]:
class PubmedRecord:
  """from https://entrezpy.readthedocs.io/en/master/tutorials/extending/pubmed.html
  Simple data class to store individual Pubmed records. Individual authors will
  be stored as dict('lname':last_name, 'fname': first_name) in authors.
  Citations as string elements in the list citations. """

  def __init__(self):
    self.pmid = None
    self.title = None
    self.abstract = None
    self.authors = []
    self.references = []

In [ ]:
#%pip install entrezpy
#import entrezpy.efetch.efetcher

# e = entrezpy.efetch.efetcher.Efetcher(tool,
#                                       email,
#                                       apikey=None,
#                                       apikey_var=None,
#                                       threads=None,
#                                       qid=None)
# analyzer = e.inquire({'db' : 'pubmed',
#                       'id' : [17284678, 9997],
#                       'retmode' : 'text',
#                       'rettype' : 'abstract'})
# print(analyzer.count, analyzer.retmax, analyzer.retstart, analyzer.uids)

In [25]:
def fetch_pubmed_abstracts(query="brain regions", max_results=100):
    handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
    record = Entrez.read(handle)
    id_list = record["IdList"]
    
    abstracts = []
    for pmid in id_list:
        fetch_handle = Entrez.efetch(db="pubmed", id=pmid, retmode="xml")
        fetch_record = Entrez.read(fetch_handle)
        abstract_text_list = fetch_record['PubmedArticle'][0]['MedlineCitation']['Article'].get('Abstract', {}).get('AbstractText', [])
        abstract_text = " ".join(abstract_text_list)
        abstracts.append(abstract_text)
    return abstracts

pubmed_medical = fetch_pubmed_abstracts("caudate")

In [31]:
pubmed_medical[0]

"Neuropathological and biomarker evidence implicates tau dysregulation as a downstream component of Huntington's disease (HD) pathobiology, yet its in vivo distribution has not been characterised using second-generation tau-PET tracers. We aimed to define the regional organisation, stage dependence and clinical relevance of tau-sensitive PET signal across the HD disease spectrum. Fifty-four participants (13 healthy controls, 9 premanifest mutation carriers and 32 manifest carriers) underwent 60-minute dynamic [¹⁸F]PI-2620 PET imaging. Tau-PET signal was quantified using distribution volume ratios (DVR) derived from reference-tissue kinetic modelling. Analyses combined region-of-interest and whole-brain mapping with threshold-based positivity profiling, modelling of cumulative genetic disease burden (CAP), and clinico-anatomical association analyses. Tau-PET abnormalities showed a spatially ordered pattern dominated by subcortical involvement. The globus pallidus exhibited the strongest

Fetch articles for different brain regions, then create an ontology and train the model

ner tags

Measure calculations, based on the notebook from class

In [32]:
import numpy as np


def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],
    }